# 2 · Mood to Color: use the whole probability distribution

## The problem

Type a feeling, get a color. A lookup table ("anger = red, sad = blue") works for single words but falls apart on
*"a misty forest at night"* or *"royal and luxurious"*: real moods are often **mixtures**, and a table can only return
one fixed answer.

## Why Jev

Jev is a **System One** model: it returns **typed judgments with calibrated probabilities**, in about 100 ms, for a
fraction of a cent. A single `Choice` question over a named palette returns not just a winner but a probability for
**every** color. Because those probabilities are calibrated, we can use them directly as **mixing ratios**: 60% gold +
40% purple is a real, paintable color.

The honest flip side, which this notebook shows on purpose: when a mood is **clear-cut** ("pure rage"), a calibrated
model puts nearly all of the probability on one color, and the "blend" is just that color. That's not a bug; it's the
model being sure.

## What we'll build

```
"royal and luxurious" ──► Jev: ONE Choice over 15 colors ──► top 3 probabilities ──► weighted RGB mix ──► swatch
                            gold    0.6                         gold 60%
                            purple  0.4                         purple 40%
                            ...                                 ...
```

## What you'll learn
- The `Choice` question type: `.choice`, `.confidence`, and the full `.probabilities` dict.
- How to **use the whole distribution**, not just the winner (the "argmax").
- How to read a distribution honestly: sharp (one clear answer) versus spread (a genuine mix).

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key from the `.env` file at the repo root (`TYPESAFE_API_KEY`),
- creates the Jev client `jev`. Every judgment in this notebook goes through it,
- imports the Jev question types (this notebook only needs `Choice`),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so numbers are easy to eyeball.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder (TYPESAFE_API_KEY)
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=24):
    """A unicode progress bar for a 0..1 value."""
    filled = round(p * width)
    return "█" * filled + "·" * (width - filled)


print("client ready")

client ready


## Step 1 · The palette

Each color has two parts:
- an **RGB value**, which is what we paint with, and
- a **description of what it evokes**, which is what Jev matches the text against.

The descriptions matter more than they look. "blue" is described as *calm, trust, cool, sadness*, so sad texts will
lean blue. Change a description and you change the model's behavior.

In [2]:
PALETTE = {
    "red":     ((214, 48, 49),  "anger, passion, heat, danger"),
    "orange":  ((230, 126, 34), "energy, warmth, autumn"),
    "gold":    ((212, 175, 55), "luxury, victory, royalty"),
    "yellow":  ((241, 196, 15), "joy, optimism, sunshine"),
    "green":   ((39, 174, 96),  "calm, nature, growth, envy"),
    "teal":    ((0, 160, 160),  "fresh, clean, modern"),
    "blue":    ((52, 120, 219), "calm, trust, cool, sadness"),
    "navy":    ((34, 49, 110),  "depth, night, serious"),
    "purple":  ((142, 68, 200), "mystery, creativity, royalty"),
    "pink":    ((240, 120, 180),"playful, sweet, love"),
    "brown":   ((140, 90, 50),  "earthy, cozy, rustic"),
    "gray":    ((130, 130, 130),"neutral, dull, boredom"),
    "black":   ((35, 35, 40),   "darkness, spooky, elegant, grief"),
    "white":   ((238, 238, 238),"purity, clean, emptiness"),
    "crimson": ((160, 20, 40),  "intensity, alarm, drama"),
}
print(len(PALETTE), "colors:", ", ".join(PALETTE))

15 colors: red, orange, gold, yellow, green, teal, blue, navy, purple, pink, brown, gray, black, white, crimson


## Step 2 · Define the question: one `Choice` over the palette

**New Jev concept: `Choice`.** A `Choice` asks "which ONE of these labels fits?". `criteria` is a dict of
`label → description`. The answer has three parts:

| Field | What it is |
|---|---|
| `.choice` | the winning label, e.g. `"gold"` |
| `.confidence` | how sure Jev is about that winner (0..1) |
| `.probabilities` | a dict with a probability for **every** label; they add up to 1 |

Most apps only read `.choice`. This notebook is about the third field.

This cell only **defines** the question. Nothing is sent to Jev yet.

In [3]:
color_question = Choice(
    instructions="Which color best matches the feeling or scene in this text?",
    criteria={name: desc for name, (_, desc) in PALETTE.items()},
)
print("criteria labels:", list(color_question.criteria))

criteria labels: ['red', 'orange', 'gold', 'yellow', 'green', 'teal', 'blue', 'navy', 'purple', 'pink', 'brown', 'gray', 'black', 'white', 'crimson']


## Step 3 · Ask once and read the full distribution

We ask about two texts on purpose: one **clear-cut** mood and one **mixed** mood. The `state` is just the text. For
each, we print the winner and the top 5 probabilities.

In [4]:
for text in ["pure, unstoppable rage", "royal and luxurious"]:
    r = jev.system_one(state=text, questions={"color": color_question})
    c = r.choices["color"]
    print(f'"{text}"  ->  winner: {c.choice}  (confidence {c.confidence:.2f})')
    for name, p in sorted(c.probabilities.items(), key=lambda kv: kv[1], reverse=True)[:5]:
        print(f"    {name:<8} {bar(p, 20)} {p:.2f}")
    print(f"    sum of all {len(c.probabilities)} probabilities: {sum(c.probabilities.values()):.2f}\n")

"pure, unstoppable rage"  ->  winner: red  (confidence 0.99)
    red      ████████████████████ 0.99
    crimson  ···················· 0.01
    orange   ···················· 0.00
    gold     ···················· 0.00
    yellow   ···················· 0.00
    sum of all 15 probabilities: 1.00

"royal and luxurious"  ->  winner: gold  (confidence 0.59)
    gold     ████████████········ 0.62
    purple   ████████············ 0.38
    red      ···················· 0.00
    orange   ···················· 0.00
    yellow   ···················· 0.00
    sum of all 15 probabilities: 1.00



**What to notice**
- "pure, unstoppable rage" is **sharp**: red takes almost all of the probability. Blending its top 3 would still be red.
- "royal and luxurious" is **spread**: gold and purple share the probability, because both are described with *royalty*. Here a blend means something.
- In both cases the probabilities add up to 1: a `Choice` divides one unit of belief across all the labels.

## Step 4 · Turn the distribution into paint

`mood_color()` is the whole feature, and the "AI" part is a single line. The rest is plain arithmetic:

1. Take the top `TOP_K` colors (3).
2. **Renormalise**: divide each by their sum, so the three weights add up to exactly 1. For example 0.55 / 0.35 / 0.05 becomes roughly 0.58 / 0.37 / 0.05.
3. Mix the RGB channels by weight: `R = Σ weightᵢ × Rᵢ`, and the same for G and B.

It returns the mixed RGB, the top colors with their renormalised weights, and a label: `"single"` if one color holds
at least `SINGLE` (90%) of the weight, otherwise `"blend"`. The label lets the UI be honest about what it's showing.

In [5]:
TOP_K = 3        # how many colors to mix
SINGLE = 0.90    # if the top color holds at least this share, call it a single color, not a blend


def mood_color(text, client):
    if "color_question" not in globals():
        raise RuntimeError("color_question is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state=text, questions={"color": color_question})
    probs = r.choices["color"].probabilities
    top = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    total = sum(p for _, p in top) or 1.0
    weights = [(name, p / total) for name, p in top]       # renormalised, sums to 1

    mix = [0.0, 0.0, 0.0]
    for name, w in weights:
        rgb = PALETTE[name][0]
        for i in range(3):
            mix[i] += rgb[i] * w
    kind = "single" if weights[0][1] >= SINGLE else "blend"
    return tuple(round(v) for v in mix), weights, kind


print("mood_color() defined. TOP_K =", TOP_K, " SINGLE =", SINGLE)

mood_color() defined. TOP_K = 3  SINGLE = 0.9


## Step 5 · Run it on clear-cut and mixed moods

> **Workshop note:** the first version of this notebook used only clear-cut phrases ("pure rage", "green with envy",
> "a warm autumn sunset"). Jev was ~100% sure about each one, so every "blend" was a single color and the idea never
> showed. We kept a few of those (they're honest and useful to see) and added **mixed** moods, where the probability
> genuinely spreads. Choosing examples that exercise the feature is part of building a demo.

In [6]:
CLEAR = ["pure, unstoppable rage", "green with envy", "a lavender field at dusk", "spooky halloween night"]
MIXED = ["royal and luxurious", "a jealous, simmering anger", "a misty forest at night",
         "bittersweet nostalgia", "a rainy Monday morning", "neon city lights at midnight"]

for group, texts in [("clear-cut", CLEAR), ("mixed", MIXED)]:
    print(f"--- {group} ---")
    for text in texts:
        rgb, weights, kind = mood_color(text, jev)
        parts = " + ".join(f"{n} {w:.0%}" for n, w in weights)
        print(f"{text:<30} [{kind:<6}] rgb{rgb}   {parts}")

--- clear-cut ---


pure, unstoppable rage         [single] rgb(213, 48, 49)   red 99% + crimson 1% + orange 0%


green with envy                [single] rgb(39, 174, 96)   green 100% + red 0% + orange 0%


a lavender field at dusk       [single] rgb(141, 69, 200)   purple 99% + blue 1% + red 0%


spooky halloween night         [single] rgb(35, 35, 40)   black 100% + red 0% + orange 0%
--- mixed ---


...

a jealous, simmering anger     [blend ] rgb(107, 125, 78)   green 61% + red 38% + crimson 1%


a misty forest at night        [blend ] rgb(36, 52, 104)   navy 72% + black 20% + blue 8%


bittersweet nostalgia          [blend ] rgb(88, 116, 169)   blue 72% + brown 16% + orange 12%


a rainy Monday morning         [blend ] rgb(110, 127, 153)   gray 74% + blue 26% + red 0%


neon city lights at midnight   [blend ] rgb(59, 51, 116)   navy 56% + purple 23% + black 21%


**What to notice**
- The clear-cut moods come out as **single** colors: the runner-ups get almost nothing, so the mix is the winner's own RGB.
- Most mixed moods come out as **blends**: anger + envy splits red and green, a misty forest at night mixes dark blues with black, a rainy Monday mixes gray and blue.
- The label is computed from the numbers, not assumed. If a "mixed" phrase comes out `single` on your run, Jev was simply more certain about it this time.
- Nothing here is hard-coded: "jealous" and "simmering" appear in no description, yet they land on green and red.

## Step 6 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired up with `ipywidgets`, and an HTML
renderer. `mood_color()` is unchanged.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

The renderer draws the **mixed swatch** on the left, then a strip showing each contributing color at its share of the
weight, so you can *see* whether you're looking at one color or a real blend. The caption says which.

In [8]:
def color_html(text):
    rgb, weights, kind = mood_color(text, jev)
    strip = "".join(f'<div title="{n} {w:.0%}" style="width:{w * 100:.1f}%;background:rgb{PALETTE[n][0]};"></div>'
                    for n, w in weights)
    contrib = " + ".join(f"{n} {w:.0%}" for n, w in weights)
    caption = ("single color: Jev is confident in one answer" if kind == "single"
               else f"blended from {contrib}")
    return box(f'<div style="{CARD}display:flex;align-items:center;gap:16px;">'
               f'<div style="width:110px;"><div style="height:70px;border-radius:10px;background:rgb{rgb};'
               f'box-shadow:0 2px 8px rgba(0,0,0,.15);"></div>'
               f'<div style="display:flex;height:8px;margin-top:6px;border-radius:4px;overflow:hidden;">{strip}</div></div>'
               f'<div><div style="font-size:15px;">"{text}"</div>'
               f'<div style="color:{SLATE};font-size:12px;margin-top:4px;">rgb{rgb} &nbsp; {caption}</div></div></div>')


display(HTML("".join(color_html(q) for q in ["pure, unstoppable rage", "royal and luxurious", "a misty forest at night"])))

## Step 7 · Try it yourself

Type into the box and press **Enter** or click the button (needs a live kernel). Good ones to try in a workshop:
- `a stormy night at sea` → mostly navy, with a little black or blue mixed in.
- `a cozy cabin in a snowstorm` → brown should lead, with white or blue mixed in.
- `a pink and gold sunrise` → pink leads with some gold: a blend you can predict from the words.
- `sheer joy` → likely a **single** yellow. Clear-cut moods give one color, and the card says so.

In [9]:
text_app("type an emotion or scene, e.g. 'a stormy night'", color_html, "Paint")

Output()

## Recap and exercises

**Key ideas**
1. A `Choice` returns a **full distribution** (`.probabilities`), not just a winner. Calibrated probabilities can be used directly as weights.
2. **Sharp vs spread**: a clear-cut input puts nearly all the weight on one label. That's information, not failure, and the UI should say so honestly.
3. **The model measures, the code decides.** Jev gave probabilities; `TOP_K`, renormalising and the RGB mix are ordinary Python.
4. Label descriptions steer the result: gold and purple both say *royalty*, which is why "royal" splits between them.

**Exercises**
- Set `TOP_K = 1` and re-run Step 5. Everything becomes a flat palette color: this is what "argmax only" looks like.
- Add a color, e.g. `"lavender": ((181, 126, 220), "gentle, dreamy, soothing")`, and try "a lavender field at dusk" again.
- Remove *royalty* from gold's description. Does "royal and luxurious" stop splitting?
- Use the same pattern elsewhere: blend a **music playlist** by genre probabilities, or weight **search filters** by intent.